# 14-2: Introduction to Survival Analysis

**Course:** Models of Statistical Analysis (MAE) — Universidad de los Andes  
**Instructor:** Prof. Alejandra Tabares  
**Week:** 14

## 1. Core Concepts in Survival Analysis

### 1.1 What Is Survival Analysis?

Survival analysis models the **time until an event of interest** (death, failure, relapse, churn, etc.). The key challenge distinguishing it from ordinary regression is **censoring**: we often do not observe the event for every subject.

### 1.2 The Survival Function

Let $T \ge 0$ be the (random) event time. The **survival function** is:

$$
S(t) = P(T > t), \quad t \ge 0
$$

- $S(0) = 1$ (everyone is event-free at the start)
- $S(\infty) = 0$ (eventually everyone experiences the event)
- $S(t)$ is monotone non-increasing and right-continuous

### 1.3 The Hazard Function

The **hazard function** (instantaneous risk rate) is:

$$
h(t) = \lim_{\Delta t \to 0} \frac{P(t \le T < t + \Delta t \mid T \ge t)}{\Delta t} = \frac{f(t)}{S(t)}
$$

Key relationships:
$$
S(t) = \exp\!\left(-\int_0^t h(u)\,du\right) = \exp(-H(t))
$$

where $H(t) = \int_0^t h(u)\,du$ is the **cumulative hazard function**.

### 1.4 Censoring

| Type | Description | Example |
|---|---|---|
| **Right censoring** | Event time is only known to exceed the observation time | Patient still alive at study end |
| **Left censoring** | Event time is only known to precede the observation time | Subject already had the event before study entry |
| **Interval censoring** | Event time falls within a known interval | Periodic follow-up visits |

**Right censoring** is the most common. An observation $(t_i, \delta_i)$ records:
- $t_i$ = observed time (event time if $\delta_i=1$, censoring time if $\delta_i=0$)
- $\delta_i$ = event indicator ($1$ = event observed, $0$ = censored)

> **Why not just ignore censored observations?** Doing so introduces **selection bias**: censored subjects may have had longer survival times, so dropping them underestimates $S(t)$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2024)
n = 200

# ── Simulate survival data with right censoring ────────────────────────────────
# Two groups: treatment (group=1) and control (group=0)
group = np.random.binomial(1, 0.5, n)

# True event times from Weibull distribution
# Treatment group has longer survival (scale = 10 vs 6)
scale_true = np.where(group == 1, 10.0, 6.0)
shape_weibull = 1.5
T_true = scale_true * np.random.weibull(shape_weibull, n)

# Censoring times from Uniform(5, 20) — independent of event time
C = np.random.uniform(5, 20, n)

# Observed time and event indicator
T_obs = np.minimum(T_true, C)
delta = (T_true <= C).astype(int)   # 1 = event observed

# A continuous covariate: age (standardized)
age = np.random.normal(55, 10, n)
age_std = (age - age.mean()) / age.std()

df = pd.DataFrame({
    'time':    T_obs,
    'event':   delta,
    'group':   group,
    'age':     age,
    'age_std': age_std
})

print('Simulated survival dataset')
print('='*45)
print(f'n = {n} subjects')
print(f'Events observed: {delta.sum()} ({100*delta.mean():.1f}%)')
print(f'Censored:        {n - delta.sum()} ({100*(1-delta.mean()):.1f}%)')
print()
print(df.head(10).to_string(index=False))

## 2. The Kaplan-Meier Estimator

The **Kaplan-Meier (KM)** estimator is a non-parametric maximum likelihood estimator of $S(t)$. It uses only the observed event times and correctly accounts for censored observations.

### Formula

Let $t_1 < t_2 < \cdots < t_k$ be the distinct observed **event** times (not censoring times). At each $t_j$:

- $d_j$ = number of events (deaths) at $t_j$
- $n_j$ = number at risk just before $t_j$ (have not yet had event or been censored)

$$
\hat{S}(t) = \prod_{j:\, t_j \le t} \left(1 - \frac{d_j}{n_j}\right)
$$

### Properties

- Step function that decreases only at observed event times
- Censored observations contribute to the risk set $n_j$ but do not cause a drop in $\hat{S}$
- **Greenwood's formula** gives the variance: $\widehat{\text{Var}}[\hat{S}(t)] = \hat{S}(t)^2 \sum_{j:\,t_j \le t} \frac{d_j}{n_j(n_j - d_j)}$
- 95% CI typically constructed on the log or log-log scale for better coverage near 0 and 1

### Log-Rank Test

To compare $\hat{S}(t)$ between two groups, the **log-rank test** uses:

$$
\chi^2 = \frac{\left(\sum_j (d_{1j} - E_{1j})\right)^2}{\sum_j V_{1j}} \sim \chi^2_1
$$

where $E_{1j}$ is the expected number of events in group 1 at time $t_j$ under $H_0: S_1(t) = S_2(t)$.

In [ ]:
# ── Kaplan-Meier: try lifelines, fall back to manual implementation ─────────────

def km_manual(times, events):
    """
    Manual Kaplan-Meier estimator.
    Returns arrays: t_unique (event times), S_hat, lower_95, upper_95
    """
    data = pd.DataFrame({'t': times, 'd': events}).sort_values('t')
    t_unique = np.sort(np.unique(data.loc[data['d'] == 1, 't']))
    n_total = len(times)

    S = 1.0
    greenwood_sum = 0.0
    S_vals, lower_vals, upper_vals = [], [], []
    n_at_risk = n_total
    prev_t = -np.inf

    for t in t_unique:
        # Update risk set: subtract those with observed time < t
        n_at_risk -= np.sum((data['t'] < t) & (data['t'] > prev_t))
        d = np.sum((data['t'] == t) & (data['d'] == 1))
        if n_at_risk > 0 and d > 0:
            S *= (1 - d / n_at_risk)
            if n_at_risk > d:
                greenwood_sum += d / (n_at_risk * (n_at_risk - d))
        se = S * np.sqrt(greenwood_sum)
        S_vals.append(S)
        lower_vals.append(max(0, S - 1.96 * se))
        upper_vals.append(min(1, S + 1.96 * se))
        prev_t = t

    return t_unique, np.array(S_vals), np.array(lower_vals), np.array(upper_vals)


try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    USE_LIFELINES = True
    print('lifelines available — using KaplanMeierFitter')
except ImportError:
    USE_LIFELINES = False
    print('lifelines not installed — using manual KM implementation')


fig, ax = plt.subplots(figsize=(9, 5))

colors = {0: 'steelblue', 1: 'darkorange'}
labels = {0: 'Control', 1: 'Treatment'}

if USE_LIFELINES:
    kmf = KaplanMeierFitter()
    for g in [0, 1]:
        mask = df['group'] == g
        kmf.fit(df.loc[mask, 'time'], df.loc[mask, 'event'], label=labels[g])
        kmf.plot_survival_function(ax=ax, ci_show=True, color=colors[g])

    # Log-rank test
    lr = logrank_test(df.loc[df['group']==0,'time'], df.loc[df['group']==1,'time'],
                      df.loc[df['group']==0,'event'], df.loc[df['group']==1,'event'])
    print(f'Log-rank test: chi2={lr.test_statistic:.3f}, p={lr.p_value:.4f}')
    ax.text(0.65, 0.85, f'Log-rank p = {lr.p_value:.4f}',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
else:
    for g in [0, 1]:
        mask = df['group'] == g
        t_u, S_u, lo, hi = km_manual(df.loc[mask,'time'].values,
                                      df.loc[mask,'event'].values)
        ax.step(np.concatenate([[0], t_u]), np.concatenate([[1], S_u]),
                where='post', color=colors[g], label=labels[g])
        ax.fill_between(np.concatenate([[0], t_u]),
                         np.concatenate([[1], lo]),
                         np.concatenate([[1], hi]),
                         step='post', alpha=0.2, color=colors[g])

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Survival probability S(t)', fontsize=12)
ax.set_title('Kaplan-Meier Survival Curves by Group\n(shaded area = 95% CI)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 3. Cox Proportional Hazards Model

The **Cox PH model** (Cox 1972) is the most widely used regression model in survival analysis. It is **semi-parametric**: the baseline hazard $h_0(t)$ is left unspecified, while the effect of covariates enters multiplicatively.

### Model

$$
h(t \mid \mathbf{x}) = h_0(t) \exp(\beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p)
$$

### Proportional Hazards Property

The ratio of hazards between two subjects with covariates $\mathbf{x}$ and $\mathbf{x}^*$ is **constant over time**:

$$
\frac{h(t \mid \mathbf{x})}{h(t \mid \mathbf{x}^*)} = \exp\!\left(\boldsymbol{\beta}^\top(\mathbf{x} - \mathbf{x}^*)\right)
$$

### Interpretation — Hazard Ratios

- $\hat{\text{HR}}_j = e^{\hat{\beta}_j}$: multiplicative change in the hazard for a one-unit increase in $x_j$, holding others fixed
- $\text{HR} > 1$: higher hazard (faster event, worse survival)
- $\text{HR} < 1$: lower hazard (slower event, better survival)
- $\text{HR} = 1$: no effect

### Partial Likelihood Estimation

Cox proposed estimating $\boldsymbol{\beta}$ by maximizing the **partial likelihood**:

$$
L(\boldsymbol{\beta}) = \prod_{i:\,\delta_i=1} \frac{\exp(\mathbf{x}_i^\top \boldsymbol{\beta})}{\sum_{j \in \mathcal{R}(t_i)} \exp(\mathbf{x}_j^\top \boldsymbol{\beta})}
$$

where $\mathcal{R}(t_i)$ is the **risk set** at time $t_i$ (all subjects still under observation). The baseline hazard $h_0(t)$ cancels out, making estimation possible without specifying it.

In [ ]:
# ── Cox PH model ───────────────────────────────────────────────────────────────

try:
    from lifelines import CoxPHFitter
    USE_LIFELINES_COX = True
    print('Fitting Cox model with lifelines.CoxPHFitter')
except ImportError:
    USE_LIFELINES_COX = False
    print('lifelines not available — using statsmodels PHReg')


if USE_LIFELINES_COX:
    cox_df = df[['time', 'event', 'group', 'age_std']].copy()
    cph = CoxPHFitter()
    cph.fit(cox_df, duration_col='time', event_col='event')
    cph.print_summary()

    # Hazard ratios with 95% CI
    hr_table = pd.DataFrame({
        'coef':      cph.params_,
        'HR':        np.exp(cph.params_),
        'HR_lower':  np.exp(cph.confidence_intervals_['95% lower-bound']),
        'HR_upper':  np.exp(cph.confidence_intervals_['95% upper-bound']),
        'p':         cph.summary['p']
    }).round(4)
    print('\nHazard Ratio Table')
    print('='*55)
    print(hr_table.to_string())

else:
    # statsmodels fallback
    from statsmodels.duration.hazard_regression import PHReg
    endog  = df['time'].values
    status = df['event'].values
    exog   = df[['group', 'age_std']].values
    cox_sm = PHReg(endog, exog, status=status,
                   exog_names=['group', 'age_std']).fit()
    print(cox_sm.summary())

    hr_table = pd.DataFrame({
        'coef':  cox_sm.params,
        'HR':    np.exp(cox_sm.params),
        'p':     cox_sm.pvalues
    }, index=['group', 'age_std']).round(4)
    print('\nHazard Ratio Table')
    print(hr_table.to_string())

In [ ]:
# ── Forest plot of hazard ratios ───────────────────────────────────────────────

if USE_LIFELINES_COX:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    cph.plot(ax=ax)
    ax.set_title('Cox PH Model — Hazard Ratios (95% CI)', fontsize=12)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.show()
else:
    # Manual forest plot
    vars_plot = ['group', 'age_std']
    hr_vals   = np.exp(cox_sm.params)
    conf      = cox_sm.conf_int()
    hr_lo     = np.exp(conf[:, 0])
    hr_hi     = np.exp(conf[:, 1])

    fig, ax = plt.subplots(figsize=(7, 3))
    y_pos = np.arange(len(vars_plot))
    ax.errorbar(hr_vals, y_pos,
                xerr=[hr_vals - hr_lo, hr_hi - hr_vals],
                fmt='o', color='steelblue', ecolor='steelblue',
                capsize=5, markersize=8)
    ax.axvline(1, color='grey', linestyle='--', linewidth=0.9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(vars_plot, fontsize=11)
    ax.set_xlabel('Hazard Ratio (log scale)', fontsize=11)
    ax.set_xscale('log')
    ax.set_title('Cox PH — Hazard Ratios (95% CI)', fontsize=12)
    plt.tight_layout()
    plt.show()

## 4. Checking the Proportional Hazards Assumption

The PH assumption — that hazard ratios are constant over time — must be verified before trusting Cox model results.

### Method 1: Log-Log Survival Plot

Under PH, the log-log transformed KM curves for two groups should be **parallel** (constant vertical distance):

$$
\ln(-\ln \hat{S}_g(t)) = \ln(-\ln S_0(t)) + \beta_g
$$

Non-parallel curves suggest the PH assumption is violated.

### Method 2: Schoenfeld Residuals (formal test)

**Schoenfeld residuals** for covariate $j$ at event time $t_i$:

$$
r_{ij} = x_{ij} - \hat{E}[x_j \mid \mathcal{R}(t_i)]
$$

If PH holds, Schoenfeld residuals are **uncorrelated with time**. A significant correlation (Grambsch-Therneau test) indicates time-varying effects.

### Method 3: Time-Varying Coefficients

Add an interaction $x_j \cdot g(t)$ (e.g., $g(t) = \ln(t)$) to the model. A significant interaction term signals PH violation.

### What to do when PH fails?

| Remedy | When to use |
|---|---|
| Stratify by the violating variable | Variable is a nuisance confounder |
| Add time-varying coefficient | Want to quantify the time-varying effect |
| Accelerated Failure Time (AFT) model | Parametric alternative to Cox |
| Flexible parametric model (Royston-Parmar) | Need smooth baseline hazard |


In [ ]:
# ── Log-log plot for PH assumption check ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if USE_LIFELINES:
    kmf2 = KaplanMeierFitter()
    for g in [0, 1]:
        mask = df['group'] == g
        kmf2.fit(df.loc[mask, 'time'], df.loc[mask, 'event'], label=labels[g])
        # Standard KM plot
        kmf2.plot_survival_function(ax=axes[0], ci_show=False, color=colors[g])
        # Log-log plot
        t_vals = kmf2.survival_function_.index.values[1:]
        S_vals = kmf2.survival_function_.iloc[1:, 0].values
        lls = np.log(-np.log(np.where(S_vals > 0, S_vals, 1e-10)))
        axes[1].plot(np.log(t_vals), lls, label=labels[g], color=colors[g])
else:
    for g in [0, 1]:
        mask = df['group'] == g
        t_u, S_u, lo, hi = km_manual(df.loc[mask,'time'].values,
                                      df.loc[mask,'event'].values)
        axes[0].step(np.concatenate([[0], t_u]), np.concatenate([[1], S_u]),
                     where='post', color=colors[g], label=labels[g])
        lls = np.log(-np.log(np.where(S_u > 0, S_u, 1e-10)))
        axes[1].plot(np.log(t_u), lls, label=labels[g], color=colors[g], marker='o', markersize=3)

axes[0].set_xlabel('Time', fontsize=11)
axes[0].set_ylabel('S(t)', fontsize=11)
axes[0].set_title('Kaplan-Meier Survival Curves', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.05)

axes[1].set_xlabel('log(Time)', fontsize=11)
axes[1].set_ylabel('log(-log S(t))', fontsize=11)
axes[1].set_title('Log-Log Plot\n(Parallel lines → PH holds)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].axhline(0, color='grey', linestyle=':', linewidth=0.7)

plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  If the lines in the log-log plot are approximately parallel, the')
print('  proportional hazards assumption is supported for the group variable.')

if USE_LIFELINES_COX:
    print('\nSchoenfeld residuals test (lifelines):')
    try:
        from lifelines.statistics import proportional_hazard_test
        ph_test = proportional_hazard_test(cph, cox_df, time_transform='rank')
        print(ph_test.summary.to_string())
        print('\n  p > 0.05 for each variable: no evidence of PH violation')
    except Exception as e:
        print(f'  Could not run PH test: {e}')